
# Reuse a published habitat model

**Background.** A habitat definition published with a paper is a
``.habitatmodel`` file. Whoever receives it should be able to see what
it declares (which images, which features, how many habitats, how it
was trained), know that their HABIT can read it, check that their own
data can enter its feature space, and then label their patients with
it -- without refitting anything and without the authors' code.

**Purpose.** You will simulate receiving such a file: train and save a
model in ``out/``, then treat the file as a download. You will read
what it declares, see the clear error HABIT gives for a file written by
a newer format version, see the error for patient data that lacks a
modality the model needs, and finally label a new patient with it.

**When to use.** Before applying anyone else's habitat model (or your
own, months later) to a new cohort.

**Key terms.** Full definitions are on :doc:`/tutorial/concepts`.

* **.habitatmodel** -- a ZIP archive with a JSON ``manifest.json``
  (format name and version, model id, feature columns, spec, cohort
  fingerprint, cohort-level preprocessing state, provenance) and the
  centroid matrix as ``arrays/centroids.npy``. No pickle.
* **format_version** -- the layout version of the archive. A HABIT
  reads files up to its own version and refuses newer ones with an
  explicit message.
* **feature space** -- the columns the centroids live in, here one
  subject-normalized and binned intensity per DCE phase. A new patient
  must provide every image the model's ``extract`` stage names.
* **cohort fingerprint** -- a non-identifying description of the
  training cohort (number of subjects, modalities, a digest of the
  subject ids).

The model uses the approved analysis definition, trained on subj001 +
subj002 of the 5-subject demo cohort. In real use you skip the training
cell and start from the downloaded file.


## Stand-in for the download: train and save a model
In practice this cell is someone else's work; you receive only the
file written at the end of it.
sphinx_gallery_thumbnail_number = 1



In [ ]:
import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt

from habit.contracts import HabitatModel, cohort_from_directory
from habit.datasets import fetch_demo
from habit.exceptions import CompatibilityError, ProcessingError
from habit.recipes import Study
from habit.spec import HabitatSpec, Spec, Stage
from habit.viz import plot_habitat_overlay

# Change DATA / MODALITIES / ROI to your preprocessed layout.
DATA = fetch_demo()
MODALITIES = ("pre_contrast", "LAP", "PVP", "delay_3min")
ROI = "LAP"
cohort = cohort_from_directory(DATA, modalities=MODALITIES, roi=ROI)
Path("out").mkdir(exist_ok=True)

spec = HabitatSpec(
    name="published_two_step",
    stages=(
        Stage("extract", Spec("raw", {"modalities": list(MODALITIES), "roi": ROI})),
        Stage("preprocess", Spec("winsorize", {"winsor_limits": [0.05, 0.05]})),
        Stage("preprocess2", Spec("minmax")),
        Stage("partition", Spec("kmeans", {"n_supervoxels": 30})),
        Stage("pool", Spec("pool")),
        Stage("preprocess_cohort", Spec("binning", {"n_bins": 10, "bin_strategy": "uniform"})),
        Stage(
            "fit",
            Spec(
                "kmeans",
                {"min_habitats": 2, "max_habitats": 10, "validation": "elbow", "n_init": 10},
            ),
        ),
        Stage("assign", Spec("nearest_centroid")),
        Stage("volume", Spec("volume")),
        Stage("msi", Spec("msi")),
        Stage("ith", Spec("ith_score")),
        Stage("graph", Spec("graph", {"include_extended_metrics": False})),
    ),
    random_seed=0,
)
Study(spec).fit_predict(cohort[:2]).save("out/published", write_maps=False)

# From here on, pretend this path is a file you downloaded.
downloaded = Path("out/published/habitat_model.habitatmodel")
print("received:", downloaded, f"({downloaded.stat().st_size / 1024:.1f} kB)")

## Load it and read what it declares
``HabitatModel.load`` reads the archive; nothing is fitted. The model
card (``summary()``) and the attributes below are everything the
authors declared. The spec travels in ``spec_payload``: it lists the
stages a new patient goes through.



In [ ]:
model = HabitatModel.load(downloaded)
print(model.summary())
print()
print("model_id:            ", model.model_id)
print("n_habitats:          ", model.n_habitats)
print("feature columns:     ", model.feature_names)
print("training subjects:   ", model.cohort_fingerprint.n_subjects)
print("training modalities: ", model.cohort_fingerprint.modalities)
print("subject-id digest:   ", model.cohort_fingerprint.subject_id_digest[:16], "...")
print("preprocessing state: ", sorted(model.preprocessing_state))
print("HABIT that wrote it: ", model.provenance.software.get("habit"))
print("spec stages:")
for stage in model.spec_payload["stages"]:
    component = stage["component"]
    print(f"  {stage['name']:<18} {component['name']:<18} {component.get('params', {})}")

## The format version lives in the manifest
``format_version`` is not an attribute of the loaded object; it is a
field of ``manifest.json`` and is checked during ``load``. Plain
``zipfile`` + ``json`` read it without HABIT.



In [ ]:
with zipfile.ZipFile(downloaded) as archive:
    print("members:", archive.namelist())
    manifest = json.loads(archive.read("manifest.json"))
print("format:        ", manifest["format"])
print("format_version:", manifest["format_version"])
print("manifest keys: ", sorted(manifest))

## A file from a newer HABIT is refused, not misread
Simulate a file written by a future HABIT: copy the archive and raise
``format_version`` by one in its manifest (stdlib only; the library
is untouched). Loading it raises ``CompatibilityError`` with an
explicit message instead of returning a model that might be read
wrongly.



In [ ]:
future = Path("out/published/future_format.habitatmodel")
with zipfile.ZipFile(downloaded) as source, zipfile.ZipFile(future, "w", zipfile.ZIP_DEFLATED) as target:
    for member in source.namelist():
        data = source.read(member)
        if member == "manifest.json":
            payload = json.loads(data)
            payload["format_version"] = int(payload["format_version"]) + 1
            data = json.dumps(payload, indent=2, sort_keys=True).encode("utf-8")
        target.writestr(member, data)
try:
    HabitatModel.load(future)
    print("loaded (unexpected)")
except CompatibilityError as error:
    print("CompatibilityError:", error)

## Your data must reach the model's feature space
The model's ``extract`` stage names four DCE phases. Suppose your
centre has no 3-minute delayed phase. First a cheap check before any
computation: compare the modalities the model was trained on with the
images your subject has. Then see what ``predict`` does anyway: it
stops with an error naming the missing modality; it never fills in or
drops a column silently.



In [ ]:
three_phases = cohort_from_directory(DATA, modalities=("pre_contrast", "LAP", "PVP"), roi=ROI)
incomplete = three_phases[2:3]
missing = sorted(set(model.cohort_fingerprint.modalities) - set(incomplete[0].images))
print(f"{incomplete[0].subject_id} lacks modalities the model needs: {missing}")
try:
    Study.from_model(model).predict(incomplete)
    print("predicted (unexpected)")
except ProcessingError as error:
    print("ProcessingError:", error)

## Label a new patient
With all four phases present, the embedded spec runs on the new
patient: subject-level normalization with the patient's own
statistics, its own 30 supervoxels, the stored bin edges, the stored
centroids. subj003 was not in the training pair.



In [ ]:
new_patient = cohort[2:3]
prediction = Study.from_model(model).predict(new_patient)
habitat_map = prediction.habitat_maps[0]
print("map model_id == file model_id:", habitat_map.model_id == model.model_id)
fractions = prediction.features.frame.set_index("subject").filter(like="_volume_fraction")
print(fractions.round(3).to_string())

fig = plot_habitat_overlay(
    new_patient[0].image("LAP"),
    habitat_map,
    title=f"{new_patient[0].subject_id}: habitats from the received model",
    crop_to="labels",
)
fig.savefig("out/published_new_patient.png", dpi=150, bbox_inches="tight")
plt.show()

## How to read the result
Measured on this run:

* The received file (a few kB) declared K = HABIT_K habitats, four feature
  columns (one per DCE phase), a training cohort of 2 subjects, the
  full stage list, and ``format_version`` 1 in its manifest.
* The copy that claims ``format_version`` 2 was refused with a
  ``CompatibilityError`` that names both versions and says to upgrade
  HABIT.
* A subject without the ``delay_3min`` phase was caught by the
  pre-check and, when predicted anyway, stopped with an error naming
  the missing modality.
* subj003 was labelled with the stored definition; its map carries the
  file's model id (True).

These checks say the file is readable and your data fit its feature
space. They do not say the habitats mean the same biology in your
cohort: scanner, protocol and population differences still need your
own validation.



## Where to go next
* Train, save and label held-out patients yourself, with a
  fresh-process check:
  :doc:`/auto_examples/01_complete/plot_18_train_save_predict`.
* Two separately fitted models need their ids matched:
  :doc:`/auto_examples/01_complete/plot_19_label_matching`.
* Export the results and write the methods paragraph:
  :doc:`/auto_examples/01_complete/plot_23_export_methods`.
* The reference analysis: :doc:`/auto_examples/01_complete/plot_01_two_step_spec`.

